# TF-IDF + Logistic Regression Model Evaluation

Evaluate the TF-IDF + Logistic Regression transaction classifier across serving strategies.

This is a scikit-learn pipeline, so ONNX conversion uses `skl2onnx`. Some strategies
(GPU EPs, quantization) may have limited benefit for a linear model -- documented below.

**Strategies**:
1. Sklearn baseline (CPU)
2. ONNX (no optimization) via skl2onnx
3. ONNX with graph optimization
4. Dynamic quantization
5. Static quantization - aggressive
6. Static quantization - conservative
7. GPU execution providers (CUDA, TensorRT)
8. OpenVINO execution provider

In [ ]:
import os
import sys
import time
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline

sys.path.insert(0, os.path.join(os.getcwd(), 'scripts'))
from benchmark_utils import (
    get_model_size_mb, benchmark_latency, benchmark_batch_throughput,
    benchmark_ort_session, print_benchmark_results, collect_result_row
)

MODEL_NAME = "tfidf_logreg"
MODEL_DIR = f"models/{MODEL_NAME}"
ONNX_DIR = f"models/{MODEL_NAME}_onnx"
os.makedirs(ONNX_DIR, exist_ok=True)

all_results = []

## Load sample data

In [ ]:
sample_texts = [
    "TESCO STORES 3456 -45.20 Friday",
    "DOMINOS #7548 -22.50 Thursday",
    "OLIVE GARDEN #9785 -67.30 Saturday",
    "AMAZON.COM -129.99 Monday",
    "SHELL OIL 12345 -55.00 Tuesday",
    "NETFLIX.COM -15.99 Wednesday",
    "WHOLE FOODS MKT -88.45 Sunday",
    "UBER *TRIP -18.75 Friday",
]

single_text = sample_texts[0]
batch_texts = sample_texts * 4  # 32 samples

print(f"Single sample: '{single_text}'")
print(f"Batch size: {len(batch_texts)}")

## 1. Sklearn Baseline (CPU)

In [ ]:
# The model may be saved as .pkl, .joblib, or via MLflow format
# Try common patterns
model_file = None
for candidate in ['model.pkl', 'model.joblib', 'pipeline.pkl', 'pipeline.joblib']:
    p = os.path.join(MODEL_DIR, candidate)
    if os.path.exists(p):
        model_file = p
        break

if model_file is None:
    # MLflow sklearn format: look for model directory
    mlflow_model_path = os.path.join(MODEL_DIR, 'model', 'model.pkl')
    if os.path.exists(mlflow_model_path):
        model_file = mlflow_model_path
    else:
        print(f"Model file not found in {MODEL_DIR}. Listing contents:")
        for root, dirs, files in os.walk(MODEL_DIR):
            for f in files:
                print(f"  {os.path.join(root, f)}")
        raise FileNotFoundError(f"Cannot find sklearn model in {MODEL_DIR}")

pipeline = joblib.load(model_file)
model_size = get_model_size_mb(MODEL_DIR)
print(f"Loaded model from {model_file}")
print(f"Model size: {model_size:.2f} MB")

In [ ]:
def predict_sklearn_single(text):
    return pipeline.predict_proba([text])

def predict_sklearn_batch(texts):
    return pipeline.predict_proba(texts)

latency_results = benchmark_latency(predict_sklearn_single, single_text)
batch_results = benchmark_batch_throughput(predict_sklearn_batch, batch_texts)
results_sklearn = {**latency_results, **batch_results}

print_benchmark_results(results_sklearn, MODEL_NAME, "sklearn_cpu", model_size)
all_results.append(collect_result_row(MODEL_NAME, "sklearn_cpu", "CPU", model_size, results_sklearn))

## 2. ONNX Export (No Optimization) via skl2onnx

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import StringTensorType
import onnxruntime as ort

onnx_model_path = os.path.join(ONNX_DIR, "model.onnx")

if not os.path.exists(onnx_model_path):
    initial_type = [('input', StringTensorType([None, 1]))]
    onnx_model = convert_sklearn(
        pipeline,
        initial_types=initial_type,
        target_opset=17
    )
    with open(onnx_model_path, "wb") as f:
        f.write(onnx_model.SerializeToString())
    print(f"ONNX model exported to {onnx_model_path}")
else:
    print(f"ONNX model already exists at {onnx_model_path}")

onnx_size = get_model_size_mb(onnx_model_path)
print(f"ONNX model size: {onnx_size:.2f} MB")

In [ ]:
session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL

ort_session_no_opt = ort.InferenceSession(
    onnx_model_path,
    sess_options=session_options,
    providers=['CPUExecutionProvider']
)

input_name = ort_session_no_opt.get_inputs()[0].name
print(f"ONNX input name: {input_name}")

def make_ort_input(texts):
    if isinstance(texts, str):
        texts = [texts]
    return {input_name: np.array(texts).reshape(-1, 1)}

sample_ort = make_ort_input(single_text)
batch_ort = make_ort_input(batch_texts)

results_onnx_no_opt = benchmark_ort_session(ort_session_no_opt, sample_ort, batch_ort)
print_benchmark_results(results_onnx_no_opt, MODEL_NAME, "onnx_no_opt", onnx_size)
all_results.append(collect_result_row(MODEL_NAME, "onnx_no_opt", "CPU", onnx_size, results_onnx_no_opt))

## 3. ONNX with Graph Optimization (ORT_ENABLE_EXTENDED)

In [ ]:
optimized_model_path = os.path.join(ONNX_DIR, "model_optimized.onnx")

session_options_opt = ort.SessionOptions()
session_options_opt.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
session_options_opt.optimized_model_filepath = optimized_model_path

ort_session_opt = ort.InferenceSession(
    onnx_model_path,
    sess_options=session_options_opt,
    providers=['CPUExecutionProvider']
)

opt_size = get_model_size_mb(optimized_model_path) if os.path.exists(optimized_model_path) else onnx_size

ort_session_opt_loaded = ort.InferenceSession(
    optimized_model_path if os.path.exists(optimized_model_path) else onnx_model_path,
    providers=['CPUExecutionProvider']
)

results_onnx_opt = benchmark_ort_session(ort_session_opt_loaded, sample_ort, batch_ort)
print_benchmark_results(results_onnx_opt, MODEL_NAME, "onnx_optimized", opt_size)
all_results.append(collect_result_row(MODEL_NAME, "onnx_optimized", "CPU", opt_size, results_onnx_opt))

## 4. Dynamic Quantization (Intel Neural Compressor)

In [ ]:
import neural_compressor
from neural_compressor import quantization

fp32_model = neural_compressor.model.onnx_model.ONNXModel(onnx_model_path)

config_dynamic = neural_compressor.PostTrainingQuantConfig(
    approach="dynamic"
)

q_model_dynamic = quantization.fit(
    model=fp32_model,
    conf=config_dynamic
)

dynamic_quant_path = os.path.join(ONNX_DIR, "model_quantized_dynamic.onnx")
q_model_dynamic.save_model_to_file(dynamic_quant_path)

dq_size = get_model_size_mb(dynamic_quant_path)
print(f"Dynamic quantized model size: {dq_size:.2f} MB")

In [ ]:
ort_session_dq = ort.InferenceSession(dynamic_quant_path, providers=['CPUExecutionProvider'])

results_dq = benchmark_ort_session(ort_session_dq, sample_ort, batch_ort)
print_benchmark_results(results_dq, MODEL_NAME, "dynamic_quant", dq_size)
all_results.append(collect_result_row(MODEL_NAME, "dynamic_quant", "CPU", dq_size, results_dq))

## 5. Static Quantization - Aggressive

Note: For a linear model (TF-IDF + LogReg), quantization benefit is typically minimal.
The model parameters are already simple linear weights. We measure it for completeness.

In [ ]:
from torch.utils.data import Dataset

class TextCalibrationDataset(Dataset):
    def __init__(self, csv_path, max_samples=256):
        df = pd.read_csv(csv_path)
        self.texts = df['payee'].dropna().unique()[:max_samples].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return ({input_name: np.array([self.texts[idx]]).reshape(1, 1)}, 0)

calib_dataset = TextCalibrationDataset('data/transactions.csv')
calib_dataloader = neural_compressor.data.DataLoader(
    framework='onnxruntime',
    dataset=calib_dataset,
    batch_size=1
)

fp32_model_agg = neural_compressor.model.onnx_model.ONNXModel(onnx_model_path)

config_static_aggressive = neural_compressor.PostTrainingQuantConfig(
    accuracy_criterion=neural_compressor.config.AccuracyCriterion(
        criterion="absolute",
        tolerable_loss=0.05
    ),
    approach="static",
    device='cpu',
    quant_level=1,
    quant_format="QOperator",
    recipes={"graph_optimization_level": "ENABLE_EXTENDED"},
    calibration_sampling_size=128
)

try:
    q_model_agg = quantization.fit(
        model=fp32_model_agg,
        conf=config_static_aggressive,
        calib_dataloader=calib_dataloader,
        eval_dataloader=calib_dataloader
    )
    agg_quant_path = os.path.join(ONNX_DIR, "model_quantized_aggressive.onnx")
    q_model_agg.save_model_to_file(agg_quant_path)
    agg_size = get_model_size_mb(agg_quant_path)
    print(f"Aggressive quantized model size: {agg_size:.2f} MB")

    ort_session_agg = ort.InferenceSession(agg_quant_path, providers=['CPUExecutionProvider'])
    results_agg = benchmark_ort_session(ort_session_agg, sample_ort, batch_ort)
    print_benchmark_results(results_agg, MODEL_NAME, "static_quant_aggressive", agg_size)
    all_results.append(collect_result_row(MODEL_NAME, "static_quant_aggressive", "CPU", agg_size, results_agg))
except Exception as e:
    print(f"Static quantization (aggressive) failed for TF-IDF+LogReg: {e}")
    print("This is expected for simple linear models with limited quantizable ops.")

## 6. Static Quantization - Conservative

In [ ]:
fp32_model_con = neural_compressor.model.onnx_model.ONNXModel(onnx_model_path)

config_static_conservative = neural_compressor.PostTrainingQuantConfig(
    accuracy_criterion=neural_compressor.config.AccuracyCriterion(
        criterion="absolute",
        tolerable_loss=0.01
    ),
    approach="static",
    device='cpu',
    quant_level=0,
    quant_format="QOperator",
    recipes={"graph_optimization_level": "ENABLE_EXTENDED"},
    calibration_sampling_size=128
)

try:
    q_model_con = quantization.fit(
        model=fp32_model_con,
        conf=config_static_conservative,
        calib_dataloader=calib_dataloader,
        eval_dataloader=calib_dataloader
    )
    con_quant_path = os.path.join(ONNX_DIR, "model_quantized_conservative.onnx")
    q_model_con.save_model_to_file(con_quant_path)
    con_size = get_model_size_mb(con_quant_path)
    print(f"Conservative quantized model size: {con_size:.2f} MB")

    ort_session_con = ort.InferenceSession(con_quant_path, providers=['CPUExecutionProvider'])
    results_con = benchmark_ort_session(ort_session_con, sample_ort, batch_ort)
    print_benchmark_results(results_con, MODEL_NAME, "static_quant_conservative", con_size)
    all_results.append(collect_result_row(MODEL_NAME, "static_quant_conservative", "CPU", con_size, results_con))
except Exception as e:
    print(f"Static quantization (conservative) failed for TF-IDF+LogReg: {e}")
    print("This is expected for simple linear models with limited quantizable ops.")

## 7. GPU Execution Providers

Note: GPU acceleration is unlikely to benefit TF-IDF+LogReg due to the model's simplicity
and the overhead of CPU-to-GPU data transfer. We measure for completeness.

In [ ]:
try:
    ort_session_cuda = ort.InferenceSession(onnx_model_path, providers=['CUDAExecutionProvider'])
    results_cuda = benchmark_ort_session(ort_session_cuda, sample_ort, batch_ort)
    print_benchmark_results(results_cuda, MODEL_NAME, "onnx_cuda", onnx_size)
    all_results.append(collect_result_row(MODEL_NAME, "onnx_cuda", "RTX 6000", onnx_size, results_cuda))
except Exception as e:
    print(f"CUDAExecutionProvider not available or not applicable: {e}")

In [ ]:
try:
    ort_session_trt = ort.InferenceSession(onnx_model_path, providers=['TensorrtExecutionProvider'])
    results_trt = benchmark_ort_session(ort_session_trt, sample_ort, batch_ort)
    print_benchmark_results(results_trt, MODEL_NAME, "onnx_tensorrt", onnx_size)
    all_results.append(collect_result_row(MODEL_NAME, "onnx_tensorrt", "RTX 6000", onnx_size, results_trt))
except Exception as e:
    print(f"TensorrtExecutionProvider not available or not applicable: {e}")

## 8. OpenVINO Execution Provider

In [ ]:
try:
    ort_session_ov = ort.InferenceSession(onnx_model_path, providers=['OpenVINOExecutionProvider'])
    results_ov = benchmark_ort_session(ort_session_ov, sample_ort, batch_ort)
    print_benchmark_results(results_ov, MODEL_NAME, "onnx_openvino", onnx_size)
    all_results.append(collect_result_row(MODEL_NAME, "onnx_openvino", "CPU (OpenVINO)", onnx_size, results_ov))
except Exception as e:
    print(f"OpenVINOExecutionProvider not available or not applicable: {e}")

## Summary

In [ ]:
results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))
results_df.to_csv(f"results/{MODEL_NAME}_evaluation.csv", index=False)
print(f"\nResults saved to results/{MODEL_NAME}_evaluation.csv")